In [18]:
import difflib
import xml.etree.ElementTree as ET
import subprocess

In [19]:
def read_file(file_path):
    with open(file_path, 'r') as file:
        return file.readlines()

def show_diff(file1_lines, file2_lines):
    differ = difflib.Differ()
    diff = list(differ.compare(file1_lines, file2_lines))
    new_diff = []
    changed_line_numbers = []
    original_line_number = 0
    modified_line_number = 0
    lines_in_modified_to_mark_as_changed = []

    for i in range(len(diff)): # We gon assume for now that only diffs are changes
        line = diff[i]
        if line.startswith('- '):
            original_line_number += 1
        elif line.startswith('+ '):
            changed_line_numbers.append(modified_line_number + 1)
            modified_line_number += 1
        elif line.startswith('? '):
            continue
        else:
            original_line_number += 1
            modified_line_number += 1
        if line.startswith('- ') and (line[1:].strip()[1:5] == "step"): # Note changed!
            new_diff.append(line)
            new_diff.append(diff[i + 1])
            new_diff.append(diff[i + 2])
            lines_in_modified_to_mark_as_changed.append(modified_line_number + 1)
    return (''.join(diff), lines_in_modified_to_mark_as_changed)

def export_to_xml(xml_lines, lines_in_modified_to_mark_as_changed, output_file):
    for line_no in lines_in_modified_to_mark_as_changed:
        # Make change appear as green note
        xml_lines.insert(line_no + 6, '   <notehead color="#31c854">normal</notehead>') 

    xml_string = '\n'.join(xml_lines)
    root = ET.fromstring(xml_string)
    tree = ET.ElementTree(root)
    with open(output_file, 'wb') as f:
        tree.write(f, encoding='utf-8', xml_declaration=True)
        
"""
Converts a MusicXML file to SVG using Verovio. 

Parameters:
- verovio_path: Absolute path to the installed verovio file 
- musicxml_path: Path to the input MusicXML file.
- svg_output_path: Path where the SVG output should be saved.
"""
def musicxml_to_svg(verovio_path, musicxml_path, svg_output_path):
    try:
        subprocess.run([
            verovio_path,  # Use the absolute path
            "-f", "musicxml",
            "-o", svg_output_path,
            musicxml_path
        ], check=True)
        print("Conversion successful.")
    except subprocess.CalledProcessError as e:
        print("Failed to convert MusicXML to SVG:", e)

In [20]:
def main():
    # Generate MusicXML Diff file
    old_file = 'musescore_sample1'
    new_file = 'musescore_sample4'
    file1_path = f'./sheet_music/{old_file}.musicxml'
    file2_path = f'./sheet_music/{new_file}.musicxml'
    xml_dff_file = f'./diffs/{old_file}_{new_file}_diff.musicxml'

    diff, lines_in_modified_to_mark_as_changed = show_diff(read_file(file1_path), read_file(file2_path))
    export_to_xml(read_file(file2_path), lines_in_modified_to_mark_as_changed, xml_dff_file)
    
    # Convert MusicXML Diff file to SVG
    verovio_path = '/opt/homebrew/bin/verovio'
    musicxml_to_svg(verovio_path, xml_dff_file, f'{xml_dff_file[:-8]}svg')
    
if __name__ == "__main__":
    main()

Conversion successful.


Output written to ./diffs/musescore_sample1_musescore_sample4_diff.svg.
